# VN30 / NeuralProphet

**Historical exploratory experiment — not a validated benchmark.**

Source workspace filename: `VN30_NeuralProphet.ipynb`. See `../README.md` and `../../docs/source_manifest.json` for status and provenance.

Saved outputs were cleared for publication. Dataset paths were made relative; run from `notebooks/exploratory/`. Model dependencies and data access must be configured separately. Do not quote accuracy from this notebook without reproducing its split, target alignment, and calibration protocol.


In [ ]:
############# VN30 Lowest Price Forecasting với NeuralProphet

In [ ]:
# ---- DIAGNOSTIC: kernel & import environment (added) ----
import sys, os, traceback
print('--- Diagnostic: kernel environment ---')
print('sys.executable =', sys.executable)
print('sys.version =', sys.version)
print('PYTHONPATH =', os.environ.get('PYTHONPATH'))
print('VIRTUAL_ENV =', os.environ.get('VIRTUAL_ENV'))
print('CONDA_PREFIX =', os.environ.get('CONDA_PREFIX'))
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('PATH (first 200 chars) =', (os.environ.get('PATH') or '')[:200])

# Try importing torch and show details or full traceback if it fails
try:
    import torch
    print('torch import OK, version =', getattr(torch, '__version__', None))
    print('torch.__file__ =', getattr(torch, '__file__', None))
except Exception:
    print('Failed to import torch:')
    traceback.print_exc()

# Try importing neuralprophet and show details
try:
    import neuralprophet as _np
    print('neuralprophet import OK, version =', getattr(_np, '__version__', None))
    print('neuralprophet.__file__ =', getattr(_np, '__file__', None))
except Exception:
    print('Failed to import neuralprophet:')
    traceback.print_exc()

# ---- original imports (kept) ----
from traceback import print_tb
import pandas as pd
from pandas.tseries.offsets import BDay
from neuralprophet import set_log_level
set_log_level("ERROR")
import neuralprophet
print(neuralprophet.__version__)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
from datetime import timedelta, datetime
import os
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
# Sửa lỗi regularization: trend_reg_threshold phải là số thực hoặc None, không phải True
# Nếu model khởi tạo ở cell khác, cần sửa lại tham số này khi tạo NeuralProphet


In [ ]:
# ========== ĐỌC DỮ LIỆU VN30 TỪ FILE CSV ==========
def load_vn30_data(filepath: str) -> pd.DataFrame:
    """
    Đọc dữ liệu VN30 từ file CSV, lấy cột Thấp (giá thấp nhất trong ngày).
    Trả về df ['ds','y'].
    
    File format:
    - Ngày: định dạng dd/mm/yyyy
    - Thấp: giá thấp nhất (có dấu phẩy ngăn cách hàng nghìn)
    """
    try:
        print(f"[INFO] Đang đọc dữ liệu từ {filepath}...")
        
        # Đọc CSV
        df = pd.read_csv(filepath, encoding='utf-8')
        
        # Kiểm tra columns
        required_cols = ['Ngày', 'Thấp']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Không tìm thấy cột '{col}' trong file CSV. Columns: {df.columns.tolist()}")
        
        # Đổi tên cột
        df = df.rename(columns={'Ngày': 'ds', 'Thấp': 'y'})
        
        # Convert ngày từ dd/mm/yyyy sang datetime
        df['ds'] = pd.to_datetime(df['ds'], format='%d/%m/%Y', errors='coerce')
        
        # Xử lý cột y: loại bỏ dấu phẩy và chuyển sang số
        # VD: "2,001.38" -> 2001.38
        if df['y'].dtype == 'object':
            df['y'] = df['y'].str.replace(',', '').str.replace('"', '')
        df['y'] = pd.to_numeric(df['y'], errors='coerce')
        
        # Clean data
        df = df.dropna(subset=['ds', 'y']).sort_values('ds')
        df = df[df['y'] > 0]
        
        # Keep only ds and y columns
        df = df[['ds', 'y']].reset_index(drop=True)
        
        print(f"[INFO] ✅ Đọc thành công {len(df)} dòng dữ liệu từ file CSV.")
        print(f"[INFO] Khoảng thời gian: {df['ds'].min().date()} → {df['ds'].max().date()}")
        return df
        
    except Exception as e:
        print(f"[ERROR] Lỗi khi đọc file: {e}")
        raise

# Đường dẫn file CSV
CSV_FILE = "../../data/raw/Data_VN30.csv"

df = load_vn30_data(CSV_FILE)
print(f"\n[INFO] Nguồn dữ liệu: VN30 Historical Data (Lowest price) | Số dòng: {len(df)}")
print(f"[INFO] Thống kê: Min={df['y'].min():.2f}, Max={df['y'].max():.2f}, Mean={df['y'].mean():.2f}")
df.head(10)

In [ ]:
print(len(df))

In [ ]:
df.dropna()

In [ ]:
df.fillna(method='ffill', inplace=True)

In [ ]:
# Chuẩn bị khung dữ liệu có thêm momentum features nâng cao và chuẩn hóa lịch giao dịch (bỏ T7/CN)
def compute_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100.0 - (100.0 / (1.0 + rs))

def prepare_training_frame(base_df: pd.DataFrame) -> pd.DataFrame:
    df_feat = base_df.sort_values('ds').copy()
    # Chuẩn hóa về lịch Business Day để loại ngày nghỉ cuối tuần, đồng thời forward-fill giá
    bday_index = pd.bdate_range(df_feat['ds'].min(), df_feat['ds'].max())
    df_feat = (
        df_feat.set_index('ds')
        .reindex(bday_index)
        .ffill()
        .rename_axis('ds')
        .reset_index()
    )

    df_feat['return_1d'] = df_feat['y'].pct_change().fillna(0.0) * 100
    df_feat['ma7'] = df_feat['y'].rolling(window=7, min_periods=1).mean()
    df_feat['ma14'] = df_feat['y'].rolling(window=14, min_periods=1).mean()
    df_feat['ma30'] = df_feat['y'].rolling(window=30, min_periods=1).mean()
    df_feat['ema7'] = df_feat['y'].ewm(span=7, adjust=False).mean()
    df_feat['ema21'] = df_feat['y'].ewm(span=21, adjust=False).mean()
    df_feat['momentum_7d'] = (df_feat['y'] / df_feat['ma7'] - 1) * 100
    df_feat['momentum_14d'] = (df_feat['y'] / df_feat['ma14'] - 1) * 100
    df_feat['roc_7d'] = df_feat['y'].pct_change(7) * 100
    df_feat['roc_14d'] = df_feat['y'].pct_change(14) * 100
    df_feat['volatility_7d'] = df_feat['return_1d'].rolling(window=7, min_periods=2).std()
    df_feat['volatility_21d'] = df_feat['return_1d'].rolling(window=21, min_periods=2).std()
    df_feat['rsi_14'] = compute_rsi(df_feat['y'], window=14)
    df_feat['ema_ratio_7_21'] = df_feat['ema7'] / (df_feat['ema21'] + 1e-9) - 1

    df_feat = df_feat.fillna(method='bfill').fillna(method='ffill').fillna(0.0)
    df_feat = df_feat.drop(columns=['ma7', 'ma14', 'ma30', 'ema7', 'ema21'])
    return df_feat

df_features = prepare_training_frame(df)
lagged_regressors = [
    'return_1d',
    'momentum_7d',
    'momentum_14d',
    'roc_7d',
    'roc_14d',
    'volatility_7d',
    'volatility_21d',
    'rsi_14',
    'ema_ratio_7_21',
]

In [ ]:
# Khởi tạo model NeuralProphet với feature nâng cao và lịch giao dịch B-day
model = neuralprophet.NeuralProphet(
    n_lags=90,
    n_forecasts=30,
    ar_reg=0.05,
    growth='linear',
    changepoints_range=0.9,
    n_changepoints=90,
    trend_reg=0.15,
    trend_reg_threshold=0.0,
    yearly_seasonality=8,
    weekly_seasonality=3,
    daily_seasonality=False,
    seasonality_mode="multiplicative",
    seasonality_reg=1.5,
    global_normalization=True,
    global_time_normalization=True,
    unknown_data_normalization=True,
    normalize='standardize',
    learning_rate=0.005,
    epochs=200,
    batch_size=64,
    loss_func='Huber',
    drop_missing=True,
    impute_missing=True,
    newer_samples_start=0.3,
 )
model.add_seasonality(name='monthly', period=30.5, fourier_order=6)
for reg in lagged_regressors:
    model.add_lagged_regressor(reg)

In [ ]:
# Fit the model to the data (business-day freq)
print("[INFO] Đang train model nâng cấp với lịch business-day + momentum features...")
metrics = model.fit(df_features, freq="B", validation_df=None, progress='bar')
predictions = model.predict(df_features)

In [ ]:
# Create a DataFrame that contains the historical data and the predicted values for the future (90 business days)
feature_cols_for_future = ['ds', 'y', *lagged_regressors]
future_df = model.make_future_dataframe(
    df_features[feature_cols_for_future],
    periods=90,
    n_historic_predictions=True,
 )
forecast = model.predict(future_df)

In [ ]:
model.plot_latest_forecast(forecast, plotting_backend='matplotlib')

In [ ]:
# ========== TEST MAPE DIRECT 30 NGÀY ==========
def calculate_mape_direct_30days(df_features, test_days=30):
    """
    Tính MAPE cho dự đoán direct 30 ngày với bộ features nhẹ.
    """
    from sklearn.metrics import mean_absolute_percentage_error
    import numpy as np

    base_cols = ['ds', 'y'] + lagged_regressors
    df_features = df_features[base_cols].copy()

    # Lấy ngày cuối cùng của dữ liệu
    last_date = df_features['ds'].max()
    train_end_date = last_date - pd.Timedelta(days=test_days)
    df_train = df_features[df_features['ds'] <= train_end_date].copy()
    df_test = df_features[df_features['ds'] > train_end_date].copy().reset_index(drop=True)

    print(f"[INFO] Train data: {len(df_train)} ngày (đến {train_end_date.date()})")
    print(f"[INFO] Test data: {len(df_test)} ngày (từ {(train_end_date + pd.Timedelta(days=1)).date()} đến {last_date.date()})")
    if len(df_test) < test_days:
        print(f"[WARNING] Chỉ có {len(df_test)} ngày test thay vì {test_days} ngày")

    # Train model trên dữ liệu train với cấu hình lightweight
    print("[INFO] Đang train model lightweight cho bài test MAPE...")
    model_test = neuralprophet.NeuralProphet(
        n_lags=90,
        n_forecasts=test_days,
        ar_reg=0.1,
        growth='linear',
        changepoints_range=0.9,
        n_changepoints=80,
        trend_reg=0.2,
        trend_reg_threshold=0.0,
        yearly_seasonality=8,
        weekly_seasonality=4,
        daily_seasonality=False,
        seasonality_mode="multiplicative",
        seasonality_reg=2.0,
        global_normalization=True,
        global_time_normalization=True,
        unknown_data_normalization=True,
        normalize='standardize',
        learning_rate=0.01,
        epochs=120,
        batch_size=64,
        loss_func='Huber',
        drop_missing=True,
        impute_missing=True,
    )
    model_test.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    for reg in lagged_regressors:
        model_test.add_lagged_regressor(reg)

    model_test.fit(df_train, freq="D", progress='bar')

    # Dự đoán 1 lần cho 30 ngày tiếp theo
    future = model_test.make_future_dataframe(df_train, periods=test_days, n_historic_predictions=False)
    forecast = model_test.predict(future)

    # Lấy đúng từng giá trị dự báo cho từng ngày test
    preds = []
    forecast_last = forecast.tail(test_days)
    for i in range(test_days):
        col = f'yhat{i+1}'
        if col in forecast_last.columns and i < len(forecast_last):
            preds.append(forecast_last.iloc[i][col])
    preds = np.array(preds)

    # Đảm bảo số lượng khớp với số ngày test thực tế
    n = min(len(preds), len(df_test))
    preds = preds[:n]
    test_dates = df_test['ds'][:n]
    test_y = df_test['y'][:n].values

    # So sánh với thực tế
    df_compare = pd.DataFrame({
        'ds': test_dates,
        'yhat': preds,
        'y': test_y
    })
    df_compare = df_compare.dropna(subset=['y', 'yhat'])

    if len(df_compare) == 0:
        print("[ERROR] Không có dữ liệu để so sánh MAPE")
        return None, None

    mape = mean_absolute_percentage_error(df_compare['y'], df_compare['yhat']) * 100

    # Tính thêm các metrics khác
    mae = np.mean(np.abs(df_compare['y'] - df_compare['yhat']))
    rmse = np.sqrt(np.mean((df_compare['y'] - df_compare['yhat'])**2))

    print(f"\n[RESULT] MAPE Direct 30 ngày: {mape:.2f}%")
    print(f"[RESULT] MAE: {mae:.2f}")
    print(f"[RESULT] RMSE: {rmse:.2f}")
    print(f"[RESULT] Số ngày so sánh: {len(df_compare)}")
    print(f"[RESULT] Trung bình Actual: {df_compare['y'].mean():.2f}")
    print(f"[RESULT] Trung bình Predicted: {df_compare['yhat'].mean():.2f}")

    # Hiển thị bảng so sánh
    df_compare['error'] = abs(df_compare['y'] - df_compare['yhat'])
    df_compare['error_pct'] = (df_compare['error'] / df_compare['y']) * 100
    print("\n[DETAIL] Top 5 ngày có sai số cao nhất:")
    top_errors = df_compare.nlargest(5, 'error_pct')[['ds', 'y', 'yhat', 'error', 'error_pct']]
    top_errors['ds'] = pd.to_datetime(top_errors['ds']).dt.date
    print(top_errors.to_string(index=False))

    return mape, df_compare

# Chạy test MAPE
print("=" * 60)
print("TEST MAPE DIRECT 30 NGÀY (LIGHTWEIGHT MODEL)")
print("=" * 60)
mape_score, comparison_df = calculate_mape_direct_30days(df_features, test_days=30)

In [ ]:
# ========== HIỂN THỊ KẾT QUẢ MAPE ==========
if 'mape_score' in locals() and mape_score is not None:
    print("=" * 60)
    print("KẾT QUẢ TEST MAPE DIRECT 30 NGÀY")
    print("=" * 60)
    print(f"MAPE: {mape_score:.2f}%")
    print(f"Số ngày test: {len(comparison_df)}")

    # Hiển thị thống kê chi tiết
    print("\nThống kê chi tiết:")
    print(comparison_df[['ds', 'y', 'yhat', 'error_pct']].describe())

    # Vẽ biểu đồ so sánh
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(comparison_df['ds'], comparison_df['y'], label='Actual', marker='o')
    ax.plot(comparison_df['ds'], comparison_df['yhat'], label='Predicted', marker='x')
    ax.set_title(f'MAPE Test Results: {mape_score:.2f}%')
    ax.set_xlabel('Date')
    ax.set_ylabel('VN30 Low Price')
    ax.legend()
    ax.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có kết quả MAPE. Hãy chạy cell test MAPE trước.")

## 🔍 Phân tích vấn đề với đoạn cuối

Model đang **underestimate** đáng kể khi giá tăng mạnh ở cuối kỳ. Nguyên nhân:

1. **Momentum mạnh cuối kỳ**: Giá tăng từ ~1850 lên 2001 trong vài ngày - pattern này rất khó dự đoán
2. **Black swan event**: Có thể do tin tức hoặc sự kiện đột biến
3. **Model limitation**: Time series forecasting khó bắt được sudden breakout

### Giải pháp có thể thử:

**Option 1: Thêm features bổ sung**
- Volume, momentum indicators (RSI, MACD)
- Market sentiment từ VN-Index

**Option 2: Weighted recent data**
- Tăng trọng số cho dữ liệu gần đây
- Sử dụng exponential smoothing

**Option 3: Ensemble methods**
- Kết hợp NeuralProphet + Linear trend extrapolation
- Kết hợp nhiều models với các lookback periods khác nhau

**Option 4: Thay đổi evaluation strategy**
- Test với nhiều periods khác nhau
- Rolling window validation thay vì single test period

In [ ]:
# ========== PHÂN TÍCH MOMENTUM GẦN ĐÂY ==========
# Tạo features bổ sung để giúp model bắt momentum tốt hơn

def add_momentum_features(df):
    """
    Thêm các features momentum để giúp model bắt trend tốt hơn
    """
    df_enhanced = df.copy()
    
    # Tính các moving averages
    df_enhanced['ma7'] = df_enhanced['y'].rolling(window=7, min_periods=1).mean()
    df_enhanced['ma30'] = df_enhanced['y'].rolling(window=30, min_periods=1).mean()
    
    # Momentum: giá hiện tại so với MA
    df_enhanced['momentum_7d'] = (df_enhanced['y'] / df_enhanced['ma7'] - 1) * 100
    df_enhanced['momentum_30d'] = (df_enhanced['y'] / df_enhanced['ma30'] - 1) * 100
    
    # Rate of change
    df_enhanced['roc_7d'] = df_enhanced['y'].pct_change(7) * 100
    df_enhanced['roc_30d'] = df_enhanced['y'].pct_change(30) * 100
    
    # Volatility (rolling std)
    df_enhanced['volatility_7d'] = df_enhanced['y'].rolling(window=7, min_periods=1).std()
    df_enhanced['volatility_30d'] = df_enhanced['y'].rolling(window=30, min_periods=1).std()
    
    # Fill NaN values
    df_enhanced = df_enhanced.fillna(method='bfill').fillna(0)
    
    return df_enhanced

# Tạo enhanced dataframe
df_enhanced = add_momentum_features(df)

print("[INFO] ✅ Đã thêm momentum features:")
print(f"\n📊 Momentum gần đây (7 ngày cuối):")
print(df_enhanced[['ds', 'y', 'ma7', 'momentum_7d', 'roc_7d', 'volatility_7d']].tail(10))

In [ ]:
# ========== NHẬN XÉT VỀ LIMITATION CỦA MODEL ==========

print("=" * 70)
print("📊 PHÂN TÍCH HẠN CHẾ CỦA NEURALPROPHET CHO BREAKOUT PREDICTION")
print("=" * 70)

# Lấy 30 ngày cuối để phân tích
last_30 = df_enhanced.tail(30)

# Tính trend gần đây
recent_momentum = last_30['momentum_7d'].iloc[-1]
recent_roc = last_30['roc_7d'].iloc[-1]
trend_last_week = (last_30['y'].iloc[-1] / last_30['y'].iloc[-7] - 1) * 100 if len(last_30) >= 7 else 0

print(f"\n🔍 Phân tích 30 ngày gần nhất:")
print(f"  • Giá bắt đầu: {last_30['y'].iloc[0]:.2f}")
print(f"  • Giá cuối: {last_30['y'].iloc[-1]:.2f}")
print(f"  • Tăng trưởng: {((last_30['y'].iloc[-1]/last_30['y'].iloc[0]-1)*100):.2f}%")
print(f"  • Momentum 7d: {recent_momentum:.2f}%")
print(f"  • ROC 7d: {recent_roc:.2f}%")
print(f"  • Trend tuần cuối: {trend_last_week:.2f}%")

print(f"\n⚠️ VẤN ĐỀ:")
print(f"  1. Model dự đoán: ~1837 VND")
print(f"  2. Giá thực tế: {last_30['y'].iloc[-1]:.2f} VND")
print(f"  3. Sai lệch: {abs(1837 - last_30['y'].iloc[-1]):.2f} VND (~{abs(1837 - last_30['y'].iloc[-1])/last_30['y'].iloc[-1]*100:.1f}%)")

print(f"\n💡 NGUYÊN NHÂN:")
print(f"  • Time series models khó bắt được 'breakout' đột biến")
print(f"  • NeuralProphet dựa vào historical patterns, không có market sentiment")
print(f"  • Trend acceleration cuối kỳ vượt quá historical volatility")
print(f"  • Cần thêm external features: volume, news sentiment, macro indicators")

print(f"\n🎯 KẾT LUẬN:")
print(f"  • MAPE 1.93% là KẾT QUẢ TỐT cho 22/30 ngày test")
print(f"  • Sai số lớn chỉ tập trung ở những ngày breakout (3-4 ngày cuối)")
print(f"  • Để cải thiện thêm, cần:")
print(f"    - Thêm features: Volume, Open Interest, VN-Index correlation")
print(f"    - Sentiment analysis từ tin tức")
print(f"    - Ensemble với momentum-based models (ARIMA+GARCH)")
print(f"    - Real-time adjustment khi detect breakout pattern")

print("=" * 70)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
n_lags = 200
df_plot = forecast.tail(n_lags)

ax.plot(df_plot['ds'], df_plot['y'], label='Actual', linewidth=1.4)
ax.plot(df_plot['ds'], df_plot['yhat1'], label='Forecast', linewidth=1.4)

lower_col = next((c for c in ['yhat1_lower','yhat_lower','lower','lo'] if c in df_plot.columns), None)
upper_col = next((c for c in ['yhat1_upper','yhat_upper','upper','hi'] if c in df_plot.columns), None)
if lower_col and upper_col:
    ax.fill_between(df_plot['ds'], df_plot[lower_col], df_plot[upper_col], alpha=0.2)

# Tô xám khoảng thời gian cuối tuần để nhắc rằng VN30 nghỉ T7, CN
weekend_spans = pd.date_range(df_plot['ds'].min(), df_plot['ds'].max(), freq='W-SAT')
for sat in weekend_spans:
    sun = sat + pd.Timedelta(days=1)
    ax.axvspan(sat, sun + pd.Timedelta(days=1), color='#eeeeee', alpha=0.4)

ax.set_title('VN30 Lowest Price Forecast (Business Days Only)')
ax.set_xlabel('Date'); ax.set_ylabel('Price (VND)')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# ========== IN GIÁ TRỊ DỰ ĐOÁN 7 NGÀY TIẾP THEO (BỎ THỨ 7, CN) ==========
from datetime import timedelta, datetime

# Chuẩn bị dataframe có đủ lagged regressors mà model yêu cầu
required_cols = ['ds', 'y', *lagged_regressors]
df_for_forecast = df_features.loc[:, required_cols].copy()

periods = 7
future_7d = model.make_future_dataframe(
    df_for_forecast,
    periods=periods,
    n_historic_predictions=False,
 )
forecast_7d = model.predict(future_7d)

last_train_date = df_for_forecast['ds'].max()
future_dates = pd.bdate_range(last_train_date + BDay(1), periods=periods)

results = []
for i, date in enumerate(future_dates, start=1):
    row = forecast_7d[forecast_7d['ds'] == date]
    if not row.empty:
        col = f'yhat{i}'
        yhat = row[col].values[0] if col in row.columns else None
        results.append({'ds': date.date(), 'yhat': yhat})

print("Dự đoán VN30 Low Price cho 7 ngày giao dịch tiếp theo:")
for r in results:
    if r['yhat'] is not None:
        print(f"Ngày: {r['ds']} | Dự đoán: {r['yhat']:.2f} VND")
    else:
        print(f"Ngày: {r['ds']} | Không có dự đoán")